In [ ]:
import jax
jax.config.update("jax_enable_x64", True)

In [ ]:
from astropy.io import fits
import os
import numpy as np
import jax.numpy as jnp

import tensorflow_probability.substrates.jax as tfp
tfd = tfp.distributions
tfb = tfp.bijectors

import matplotlib.pyplot as plt

In [ ]:
from gigalens.jax.scene import Component, Plane, LensModel
from gigalens.jax.profiles.mass.epl import EPL
from gigalens.jax.profiles.mass.shear import Shear
from gigalens.jax.profiles.mass.nfw import NFW_ELLIPSE, NFW_ELLIPSE_EINSTEIN
from gigalens.jax.profiles.mass.nfw_ellipse_slope import NFW_ELLIPSE_SLOPE
from gigalens.jax.profiles.mass.piemd import DPIE

from gigalens.jax.profiles.light.sersic import SersicEllipse
from gigalens.jax.profiles.light.shapelets import Shapelets

from gigalens.jax.cosmo import wCDM_Cosmo

from gigalens.jax.scene_prob_model import Dataset, ProbModel
from gigalens.simulator import SimulatorConfig

In [ ]:
from gigalens.jax.grouped_priors import DiskEllipticity
from gigalens.jax.experimental.adaptive_supersample import AdaptiveImageData, plot_factor_map

In [ ]:
def tNCDF_bij(low, high):
    return tfb.Chain([tfb.Shift(low), tfb.Scale(high-low), tfb.NormalCDF()])

class UniformBij(tfd.Uniform):
    def __init__(self, *args, event_space_bijector_class=tNCDF_bij, **kwargs):
        self._esb = event_space_bijector_class(*args)
        super().__init__(*args, **kwargs)
    def _default_event_space_bijector(self):
        return self._esb

class TruncatedNormalBij(tfd.TruncatedNormal):
    def __init__(self, *args, event_space_bijector_class=tNCDF_bij, **kwargs):
        args = [jnp.float64(arg) for arg in args]
        low, high = args[2], args[3]
        self._esb = event_space_bijector_class(low, high)
        super().__init__(*args, **kwargs)
    def _default_event_space_bijector(self):
        return self._esb

In [ ]:
#* MASS PRIORS
NFW0 = Component(NFW_ELLIPSE_SLOPE(), dict(
   theta_E = tfd.Normal(13, 1),# alpha_Rs = tfd.Uniform(10,40),
    s_E = tfd.Uniform(0,0.75),
   e1 = tfd.TruncatedNormal(0, 0.05, -0.2, 0.2),
   e2 = tfd.TruncatedNormal(0, 0.05, -0.2, 0.2),
   center_x = tfd.Normal(5.344, 0.05),
   center_y = tfd.Normal(3.805, 0.05)
))

EPL_Le = Component(EPL(18), {
   'theta_E' : tfd.TruncatedNormal(2.4, 0.1, 1, 3),
   'gamma' : tfd.TruncatedNormal(2.2, 0.5, 1, 3),
    ('e1', 'e2') : DiskEllipticity(e_max=0.3, scale=0.1),
   # e1 = TruncatedNormalBij(0, 0.1, -0.5, 0.5),
   # e2 = TruncatedNormalBij(0, 0.1, -0.5, 0.5),
   'center_x' : tfd.Normal(-22.1, 0.1),
   'center_y' : tfd.Normal(-24.7, 0.1)
})


DPIE_Ld = Component(DPIE(), {
    'theta_E' : tfd.TruncatedNormal(1.6730331, 0.1, 1, 2.5),
    'r_core' : tfd.Uniform(0,1),
    'r_cut' : tfd.Uniform(1,20),
    'center_x' : tfd.Normal(11.80977389, 0.1),
    'center_y' : tfd.Normal(23.0283886, 0.1),
    ('e1','e2') : DiskEllipticity(e_max=0.3, scale=0.05),
    # e1 = tfd.TruncatedNormal(0, 0.05, -0.3, 0.3),
    # e2 = tfd.TruncatedNormal(0, 0.05, -0.3, 0.3),
})
#* Comes along with source 3
# EPL_Ld = dict(
#     theta_E = tfd.TruncatedNormal(1.6730331, 0.1, 1, 2.5),
#     gamma = tfd.Uniform(1,3),
#     e1 = tfd.TruncatedNormal(0, 0.1, -0.3, 0.3),
#     e2 = tfd.TruncatedNormal(0, 0.1, -0.3, 0.3),
#     center_x = tfd.Normal(11.80977389, 0.1),
#     center_y = tfd.Normal(23.0283886, 0.1)
# )

# EPL_Lf = Component(EPL(50), {
#    'center_x' : tfd.Normal(-15.10088063, 0.1),
#    'center_y' : tfd.Normal(-4.66657821, 0.1),
#     ('e1', 'e2') : DiskEllipticity(e_max=0.3, scale=0.1),
#    # e1 = TruncatedNormalBij(0, 0.1, -0.5, 0.5),
#    # e2 = TruncatedNormalBij(0, 0.1, -0.5, 0.5),
#    'theta_E' : tfd.TruncatedNormal(0.8151327, 0.05, 0.2, 1.5),
#    'gamma' : tfd.TruncatedNormal(2.2266, 0.5, 1, 3)
# })


DPIE_Lf = Component(DPIE(), {
   'theta_E' : tfd.TruncatedNormal(0.8151327, 0.05, 0.2, 1.5),
   'r_core' : tfd.Uniform(0,1),
   'r_cut' : tfd.Uniform(1,20),
   'center_x' : tfd.Normal(-15.10088063, 0.1),
   'center_y' : tfd.Normal(-4.66657821, 0.1),
    ('e1', 'e2') : DiskEllipticity(e_max=0.3, scale=0.1),
   # e1 = tfd.TruncatedNormal(0, 0.05, -0.3, 0.3),
   # e2 = tfd.TruncatedNormal(0, 0.05, -0.3, 0.3),
})

shear = Component(Shear(), dict(
    gamma1 = tfd.TruncatedNormal(0., 0.1, -0.3, 0.3),
    gamma2 = tfd.TruncatedNormal(0., 0.1, -0.3, 0.3),
))

src1 = Component(Shapelets(n_max=8, use_lstsq=True), dict(
    center_x = tfd.Normal(7.67187389, 2),
    center_y = tfd.Normal(3.31911655, 2),
    beta = tfd.LogNormal(jnp.log(0.4), 0.15),
))

# src2 = Component(SersicEllipse(use_lstsq=True), dict(
#     center_x = tfd.Normal(10, 2),
#     center_y = tfd.Normal(3., 2),
#     R_sersic = tfd.LogNormal(jnp.log(0.4), 0.15),
#     n_sersic = tfd.Uniform(0.5,15),
#     e1 = tfd.Normal(0,0.1),
#     e2 = tfd.Normal(0,0.1),
# ))

#* LIGHT PRIORS
src3 = Component(Shapelets(n_max=8, use_lstsq=True), { #default n_max=12
    'center_x' : tfd.Normal(5., 1),
    'center_y' : tfd.Normal(5., 1),
    'beta' : tfd.LogNormal(jnp.log(0.4), 0.15)
})

src4 = Component(Shapelets(n_max=8, use_lstsq=True), dict( #default n_max=12
    center_x = tfd.Normal(3.7, 1),
    center_y = tfd.Normal(3.2, 1),
    beta = tfd.LogNormal(jnp.log(0.4), 0.15),
))

src5 = Component(Shapelets(n_max=6, use_lstsq=True), dict(
    center_x = tfd.Normal(3.0, 1),
    center_y = tfd.Normal(0., 1),
    beta = tfd.LogNormal(jnp.log(0.1), 0.15),
))

src9 = Component(SersicEllipse(use_lstsq=True), { #default n_max=12
    'center_x' : tfd.Normal(-10, 1),
    'center_y' : tfd.Normal(-16, 1),
    'n_sersic' : tfd.Uniform(0.1,10),
    'R_sersic' : tfd.LogNormal(jnp.log(0.4), 0.15),
    ('e1', 'e2') : DiskEllipticity(e_max=0.3, scale=0.1),
    # e1 = tfd.TruncatedNormal(0, 0.1, -0.3, 0.3),
    # e2 = tfd.TruncatedNormal(0, 0.1, -0.3, 0.3),
})

In [ ]:
#* All the physical stuff
z1_2=0.962
z3=1.166
z4_5=1.432
z9=1.506
z12_13=3.086
z8=3.549
z11=4.090

z_lens = 0.49


# Om0_prior = tfd.Uniform(0, 1)
# w0_prior = tfd.Uniform(-2, -1/3)
cosmo = Component(wCDM_Cosmo(z_lens=z_lens, z_source_ref=z4_5), dict(H0=70.0, Om0=0.3, k=0.0, w0=-1.0))

model = LensModel([
    Plane(redshift=z_lens, mass=[NFW0, DPIE_Ld, EPL_Le, DPIE_Lf, shear]),       # the one deflector
    Plane(redshift=z1_2, light=[src1]),    # nearer source plane
    Plane(redshift=z3, light=[src3]),
    Plane(redshift=z4_5, light=[src4, src5]),
    Plane(redshift=z9, light=[src9]),
    # Plane(redshift=z12, light=[src12, src13]),
    # Plane(redshift=z8, light=[src8]),
    # Plane(redshift=z11, light=[src11]),
], cosmo=cosmo)

In [ ]:
def dataset_from_dir(path, ext):

    img_path = os.path.join(path, f"source{ext}.fits")
    with fits.open(img_path) as hdul:
        observed_image = jnp.array(hdul['DATA'].data.astype("float64"))

        error_map = jnp.array(np.sqrt(hdul['STAT'].data.astype("float64")))
        # background_rms = hdul['DATA'].header['BKG_RMS']
        # exp_time = hdul['PRIMARY'].header['EXPTIME']
            # if centroids is None: (self.centroids_x, self.centroids_y) = Table(hdul['CENTROIDS'].data)['centroid'].data.T
            # if centroids_error is None: self.centroids_error = Table(hdul['CENTROIDS'].data)['sky_covariance'].data
        psf = hdul['PSF'].data.astype(jnp.float64)
        mask = hdul['MASK'].data.astype(jnp.bool)
        # hot_pix = jnp.load(os.path.join(path, f"hot_pix.npy"))

    # mask = jnp.logical_and(mask, hot_pix)

    return observed_image, error_map, psf, mask

path= "newnewcutouts/"
def ds(ext, sees):
    observed_image, error_map, psf, mask = dataset_from_dir(path, ext)

    cfg = SimulatorConfig(delta_pix=0.2, num_pix=300, supersample=1, kernel=psf, likelihood_precision="float64", conv_precision="float32")
    dset = AdaptiveImageData(observed_image, cfg, error_map=error_map, mask=mask, sees=sees)
    return dset
    

d1 = ds("1", sees=[src1])
d3 = ds("3", sees=[src3])
d4_5 = ds("4-5", sees=[src4, src5])
d9 = ds("9", sees=[src9])
# d12_13 = ds("12-13", sees=[src12_13])
# d8 = ds("8", sees=[src8])
# d11 = ds("11", sees=[src11])

prob_model = ProbModel(model, [d1, d3, d4_5, d9], mode="lstsq")

# sim = SceneSimulator(model, cfg)

In [ ]:
plt.imshow(jnp.where(d9.mask, d9.image/d9.error_map, 0.0))
plt.colorbar()

In [ ]:
fig, _ = plot_factor_map(d1.adaptive_grid)
fig, _ = plot_factor_map(d3.adaptive_grid)
fig, _ = plot_factor_map(d4_5.adaptive_grid)
fig, _ = plot_factor_map(d9.adaptive_grid)

In [ ]:
from gigalens_research.inference_utils import (
    InferenceContext, Pipeline, MAPStage, BridgeStage, MCLMCStage, MAMSStage, SVIStage, HMCStage
)
ctx = InferenceContext.from_prob_model(prob_model)
pipeline = Pipeline(ctx, seed=42)
pipeline.add(MAPStage(num_steps=2000, n_samples=128))


# def make_diag_qz(z_best):
#     return tfd.MultivariateNormalDiag(
#         loc=jnp.asarray(z_best),
#         scale_diag=jnp.full(z_best.shape[-1], 1e-2),
#     )

# pipeline.add(BridgeStage(
#     name="diag_qz",
#     version="v1",           # bump to 'v2' if you change the scale or logic
#     requires=("z_best",),
#     produces=("qz",),
#     fn=make_diag_qz,
# ))

# pipeline.add(SVIStage(n_vi=128, num_steps=1000))
from gigalens_research.inference_utils.pipeline import PTMCLMCStage
# pipeline.add(MCLMCStage(n_chains=8, num_burnin_steps=2000, num_results=2000, debug=True, progress_bar=True, regularize_mass_matrix=True))
pipeline.add(PTMCLMCStage(beta_min=0.36, n_walkers=4, debug=True))
# pipeline.add(MAMSStage(n_chains=64, num_burnin_steps=1000, num_results=1000, debug=True, progress_bar=True, regularize_mass_matrix=True))

artifacts = pipeline.run(out_dir="debug_carousel/1_2_3_4_5_9", resume=True)

In [ ]:
from gigalens_research.plotting import PosteriorReport, PipelineReport
pipeline_report = PipelineReport(pipeline)
# fig = pipeline_report.diagnostics("mclmc", chain=3)
fig = pipeline_report.diagnostics("ptmclmc")#, chain=3)
# fig = pipeline_report.diagnostics("mams", chain=3)
fig.show()
# fig = pipeline_report.diagnostics_surrogate_corner("mclmc")
# fig = pipeline_report.diagnostics_surrogate_corner("mams")
# fig.show()

In [ ]:
# posterior = pipeline.posterior()
# ts = jax.tree.structure(posterior.z_to_x(posterior.ess))
    
# jax.tree.unflatten(ts, posterior.ess)

In [ ]:
report = PosteriorReport(pipeline.posterior())
# fig = report.image_panel()
# fig.show()
# fig = report.source_panel()
# fig.show()
report.full_report()
plt.show()

In [ ]:
# import matplotlib.pyplot as plt
# import matplotlib.colors as colors
# im = sim9.lstsq_simulate(model.to_params({}), d9.image, err_map=d9.error_map, mask=d9.mask)
# plt.imshow(im, norm=colors.PowerNorm(gamma=0.5, vmin=0), origin="lower")
# plt.colorbar()
# plt.show()

In [ ]:
# im = sim4_5.lstsq_simulate(model.to_params({}), d4_5.image, err_map=d4_5.error_map, mask=d4_5.mask)
# plt.imshow(im, norm=colors.PowerNorm(gamma=0.5, vmin=0), origin="lower")
# plt.colorbar()
# plt.show()

In [ ]:
# model.to_params({})

In [ ]:
# mask = d9.mask
# masked_img = jnp.where(mask, d9.image, 0)
# plt.imshow(masked_img,norm=colors.PowerNorm(gamma=0.5, vmin=0))
# plt.colorbar()
# plt.show()

In [ ]:
# plt.imshow(d4_5.mask)
# plt.show()

In [ ]:
# jnp.sum(d4_5.mask)
# np.unique(d4_5.mask)

In [ ]:
# from gigalens.jax.scene_simulator import SceneSimulator
# from gigalens.simulator import SimulatorConfig

